# Notebook 13 — GDELT BigQuery spike validation

**Input:** `reports/topic_modeling/12_spikes_anchors/spike_events.csv` + `reports/topic_modeling/11_lda_sbert/topics_over_time.csv`  
**Output:** `reports/topic_modeling/13_gdelt_bigquery/gdelt_validation.csv` + `gdelt_validation.html` (dual-axis: local growth-velocity vs GDELT article counts / day)

**Why:** When both curves peak in the same week, the spike is more credible against **global** media volume, not only ABC headline clustering.

**Setup:** Google Cloud project, BigQuery API enabled, `gcloud auth application-default login` (or service-account JSON), `GOOGLE_CLOUD_PROJECT` set. See README subsection *GDELT BigQuery spike validation*.


In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))


In [ ]:
import os
from IPython.display import HTML, display

from gdelt_bigquery_spike_validation import (
    DEFAULT_ANCHOR_MD,
    run_validation,
    write_anchor_gdelt_markdown,
)

project = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not project:
    display(HTML(
        "<b>Set GOOGLE_CLOUD_PROJECT</b> to your GCP project id, then re-run this cell."
    ))
else:
    spikes_path = os.path.join(PROJECT_ROOT, "reports", "topic_modeling", "12_spikes_anchors", "spike_events.csv")
    out_csv = os.path.join(PROJECT_ROOT, "reports", "topic_modeling", "13_gdelt_bigquery", "gdelt_validation.csv")
    out_html = os.path.join(PROJECT_ROOT, "reports", "topic_modeling", "13_gdelt_bigquery", "gdelt_validation.html")
    anchor_md = os.path.join(PROJECT_ROOT, DEFAULT_ANCHOR_MD)
    df, html_path = run_validation(
        spikes_path=spikes_path,
        tot_path=os.path.join(PROJECT_ROOT, "reports", "topic_modeling", "11_lda_sbert", "topics_over_time.csv"),
        out_csv=out_csv,
        out_html=out_html,
        project=project,
        max_spikes=10,
        window_days=30,
        max_plots=5,
    )
    print("Rows:", len(df), "→", out_csv)
    print("Charts →", html_path)
    md_path = write_anchor_gdelt_markdown(project, spikes_path, anchor_md)
    print("Anchor GDELT summary →", md_path)
    display(df.head(10))
